In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Locate the repo root without importing from src yet.
_current = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate
    for candidate in (_current, *_current.parents)
    if (candidate / "AGENTS.md").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_selection.data_loading import load_split
from src.feature_selection.elastic_net import run_elastic_net


In [3]:
X_train, y_train = load_split("train", processed_dir=PROJECT_ROOT / "data" / "processed")
X_val, y_val = load_split("validation", processed_dir=PROJECT_ROOT / "data" / "processed")

feature_cols = list(X_train.columns)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)


Train: (1895, 25)
Validation: (600, 25)


In [4]:
# Scaler fit on TRAIN ONLY, validation only ever gets .transform().
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=feature_cols,
    index=X_train.index,
)
X_val_scaled = pd.DataFrame(
    scaler.transform(X_val),
    columns=feature_cols,
    index=X_val.index,
)


In [5]:
# Reference: smallest alpha that would zero out every coefficient
# (train only -- just for picking a sensible alpha grid).
alpha_max = np.max(
    np.abs(X_train_scaled.T @ y_train)
) / len(X_train_scaled)

print("alpha_max:", alpha_max)


alpha_max: 0.004199373087303637


In [7]:
alpha_values = [
    0.0001, 0.0005, 0.0010, 0.0015, 0.0020, 0.0025, 0.0030,
    0.0035, 0.0040, 0.0045, 0.0050, 0.0055, 0.0060, 0.0065, 0.0068,
]

lasso_results = [
    run_elastic_net(
        X_train_scaled, X_val_scaled, y_train, y_val,
        alpha=alpha,
        l1_ratio=1.0,  # l1_ratio=1.0 -> pure Lasso
    )
    for alpha in alpha_values
]

lasso_results_df = (
    pd.DataFrame(lasso_results)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

lasso_results_df


,alpha,l1_ratio,n_selected_features,selected_features,MSE,RMSE,MAE,R2
0,0.0045,1.0,0,[],0.011053,0.105132,0.077442,-0.008612
1,0.0050,1.0,0,[],0.011053,0.105132,0.077442,-0.008612
2,0.0055,1.0,0,[],0.011053,0.105132,0.077442,-0.008612
3,0.0060,1.0,0,[],0.011053,0.105132,0.077442,-0.008612
4,0.0065,1.0,0,[],0.011053,0.105132,0.077442,-0.008612
5,0.0068,1.0,0,[],0.011053,0.105132,0.077442,-0.008612
6,0.0025,1.0,5,"[sma_5, sma_60, macd_hist, volatility_20, volu...",0.011063,0.105179,0.077465,-0.009502
7,0.0040,1.0,1,[macd_hist],0.011065,0.105192,0.077491,-0.009760
8,0.0030,1.0,5,"[sma_5, sma_60, macd_hist, volatility_20, volu...",0.011067,0.105198,0.077486,-0.009862
9,0.0020,1.0,6,"[gap, sma_5, sma_60, macd_hist, volatility_20,...",0.011088,0.105300,0.077566,-0.011833


In [8]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "embedded_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_path = OUTPUT_DIR / "lasso_results.csv"
lasso_results_df.to_csv(output_path, index=False)

print("Saved:", output_path)


Saved: /Users/yangjaehoon/Desktop/StockLens/data/processed/embedded_results/lasso_results.csv
